In [15]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import RobertaTokenizer, RobertaForSequenceClassification
from torch.optim import AdamW
import numpy as np
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

# Set device (use GPU if available, otherwise CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(torch.cuda.is_available())  # Must return True
print(torch.cuda.get_device_name(0))  # Your GPU model

Using device: cuda
True
NVIDIA GeForce RTX 2050


In [16]:
#Setting the Datasets Files
train_path = "emotion_intensity_data/train.csv"  # Path to your training data
dev_path = "emotion_intensity_data/dev.csv"   # Path to development/validation data
test_path = "emotion_intensity_data/test.csv"    # Path to test data

def load_dataset(file_path):
    try:
        df = pd.read_csv(file_path)
        print(f"Loaded {file_path} with {len(df)} samples")
        return df
    except FileNotFoundError:
        print(f"Error: The file at {file_path} was not found.")
        return None

# Load all datasets
train_df = load_dataset(train_path)
dev_df = load_dataset(dev_path)
test_df = load_dataset(test_path)

if train_df is not None:
    print("\nFirst few rows of training data:")
    print(train_df.head())
    print("\nColumns in the dataset:")
    print(train_df.columns.tolist())



Loaded emotion_intensity_data/train.csv with 2768 samples
Loaded emotion_intensity_data/dev.csv with 116 samples
Loaded emotion_intensity_data/test.csv with 2767 samples

First few rows of training data:
                        id                                               text  \
0  eng_train_track_b_00001                       Colorado, middle of nowhere.   
1  eng_train_track_b_00002  This involved swimming a pretty large lake tha...   
2  eng_train_track_b_00003        It was one of my most shameful experiences.   
3  eng_train_track_b_00004  After all, I had vegetables coming out my ears...   
4  eng_train_track_b_00005                        Then the screaming started.   

   anger  fear  joy  sadness  surprise  
0      0     1    0        0         1  
1      0     2    0        0         0  
2      0     1    0        3         0  
3      0     0    0        0         0  
4      0     3    0        1         2  

Columns in the dataset:
['id', 'text', 'anger', 'fear', 'joy',

In [17]:
class EmotionIntensityDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=128):
        self.texts = dataframe['text'].tolist()
        # Get emotion intensity columns (adjust these names based on your CSV)
        self.anger = dataframe['anger'].tolist()
        self.fear = dataframe['fear'].tolist()
        self.joy = dataframe['joy'].tolist()
        self.sadness = dataframe['sadness'].tolist()
        self.surprise = dataframe['surprise'].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        
        # Get emotion intensities for this text (0-3 scale)
        emotions = torch.tensor([
            self.anger[idx],
            self.fear[idx],
            self.joy[idx],
            self.sadness[idx],
            self.surprise[idx]
        ], dtype=torch.long)
        
        # Tokenize the text (convert to numbers)
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': emotions
        }

In [22]:
# Step 2: Initialize tokenizer
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

# Step 3: Create datasets
if train_df is not None and dev_df is not None:
    train_dataset = EmotionIntensityDataset(train_df, tokenizer)
    dev_dataset = EmotionIntensityDataset(dev_df, tokenizer)
    test_dataset = EmotionIntensityDataset(test_df, tokenizer)

    # Step 4: Create data loaders
    batch_size = 8
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    dev_loader = DataLoader(dev_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    print(f"\nTraining batches: {len(train_loader)}")
    print(f"Validation batches: {len(dev_loader)}")
    print(f"Test batches: {len(test_loader)}")

    class EmotionIntensityModel(nn.Module):
        def __init__(self):
            super(EmotionIntensityModel, self).__init__()
            
            # Use pre-trained RoBERTa model
            self.roberta = RobertaForSequenceClassification.from_pretrained('roberta-base',num_labels=5 ) # We predict 5 emotions
        
        def forward(self, input_ids, attention_mask):
            # Pass text through the model
            outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
            return outputs.logits
        
    model = EmotionIntensityModel()
    model = model.to(device)
    print(f"\nModel created and moved to {device}")


Training batches: 346
Validation batches: 15
Test batches: 346


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Model created and moved to cuda


NameError: name 'EmotionIntensityModel' is not defined